In [14]:
# ----------------------------------------------------
#  Trompo simétrico – Precesión libre (g = 0)
# ----------------------------------------------------
import numpy as np
from numba import njit
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, HTMLWriter
from mpl_toolkits.mplot3d import Axes3D      # noqa: F401

# -----------------------------
#  1. Parámetros físicos
# -----------------------------
m = 0.1          # masa [kg]
r = 0.1          # radio del disco [m]
d = 0.3          # CM-punta [m]
g = 0.0          # SIN gravedad → precesión libre

# Momentos de inercia del trompo simétrico
I0 = 0.25*m*r*r + m*d*d   # ejes transversos
Iz = 0.5*m*r*r            # eje de simetría

# -----------------------------
#  2. Condiciones iniciales
# -----------------------------
theta0    = np.deg2rad(80)    # inclinación inicial (nutación)
phi0      = 0.0               # fase de precesión
psi0      = 0.0               # giro propio

dtheta0   = 0.0               # velocidad polar inicial
w_spin    = 10.0             # giro propio ω3 (rad/s)

# ---  Constantes de movimiento ----------------------
Ppsi = Iz * w_spin   # componente de L a lo largo del eje 3
Pphi = 0.0           # ≠ Iz*ω3·cosθ0  ⇒  L NO está alineado

# Estado inicial y = [φ, ψ, θ, θ̇]
y0 = np.array([phi0, psi0, theta0, dtheta0], dtype=np.float64)

# -----------------------------
#  3. Ecuaciones de movimiento
# -----------------------------
@njit(fastmath=True)
def rhs(t, y):
    """Ecuaciones diferenciales del trompo con Pφ, Pψ constantes."""
    phi, psi, theta, dtheta = y
    s, c = np.sin(theta), np.cos(theta)

    Ip = (I0*s*s + Iz*c*c)*Iz - Iz*Iz*c*c
    phi_dot = (Iz*Pphi - Iz*c*Ppsi) / Ip
    psi_dot = ((I0*s*s + Iz*c*c)*Ppsi - Iz*c*Pphi) / Ip

    dtheta_dot = ((phi_dot**2)*s*c*(I0 - Iz)
                  - phi_dot*psi_dot*Iz*s
                  + m*g*d*s) / I0          # g = 0 ⇒ último término = 0

    return np.array([phi_dot,      # φ̇
                     psi_dot,      # ψ̇
                     dtheta,       # θ̇
                     dtheta_dot],  # θ̈
                    dtype=np.float64)

# -----------------------------
#  4. Integración numérica
# -----------------------------
t_max  = 20
t_eval = np.linspace(0.0, t_max, 300)   # 1 ms de paso registrado

sol = solve_ivp(rhs, (0.0, t_max), y0,
                method="DOP853", t_eval=t_eval,
                rtol=1e-9, atol=1e-12, max_step=1e-3)

phi, psi, theta = sol.y[0], sol.y[1], sol.y[2]
t               = sol.t

# -----------------------------
#  5. Animación 3D
# -----------------------------
fig = plt.figure(figsize=(8, 4))
ax3d = fig.add_subplot(121, projection='3d', facecolor='whitesmoke')
axT  = fig.add_subplot(122,  facecolor='whitesmoke')

# Ejes 3D
lim = d*1.1
ax3d.set_xlim([-lim, lim]); ax3d.set_ylim([-lim, lim]); ax3d.set_zlim([0, 2*lim])
ax3d.set_xlabel('x'); ax3d.set_ylabel('y'); ax3d.set_zlabel('z')
ax3d.view_init(elev=25, azim=45)

# θ(t)
axT.set_xlim([0, t_max]); axT.set_ylim([0, np.pi])
axT.set_xlabel('t [s]'); axT.set_ylabel(r'$\theta$ [rad]')

# Artistas
line3d,  = ax3d.plot([], [], [], 'r-', lw=2)
tip3d,   = ax3d.plot([], [], [], 'ko', ms=6)
trace3d, = ax3d.plot([], [], [], 'b--', lw=1, alpha=0.6)
lineT,   = axT.plot([], [], 'r-')

trace_x, trace_y, trace_z = [], [], []

def init():
    line3d.set_data_3d([], [], [])
    tip3d.set_data_3d([], [], [])
    trace3d.set_data_3d([], [], [])
    lineT.set_data([], [])
    return line3d, tip3d, trace3d, lineT

def update(i):
    # Punta del trompo en el laboratorio
    x = d*np.sin(theta[i])*np.cos(phi[i])
    y = d*np.sin(theta[i])*np.sin(phi[i])
    z = d*np.cos(theta[i])

    line3d.set_data_3d([0, x], [0, y], [0, z])
    tip3d.set_data_3d([x], [y], [z])

    trace_x.append(x); trace_y.append(y); trace_z.append(z)
    trace3d.set_data_3d(trace_x, trace_y, trace_z)

    lineT.set_data(t[:i+1], theta[:i+1])
    return line3d, tip3d, trace3d, lineT

ani = FuncAnimation(fig, update, frames=range(0, len(t), 5),
                    init_func=init, interval=20, blit=True)
plt.close()
# Para Jupyter / Colab
HTMLWriter.embed_limit = 40  # MB
from IPython.display import HTML
HTML(ani.to_html5_video())